# Neural Frame Generation (NeuralFG) on Kaggle (T4 x 2)

End-to-end learned frame interpolation adapted from RESEARCH.md section 6.3, trained ONLY on the
Kaggle-native datasets in RESEARCH.md section 3.2. Companion of `kaggle_neural_supersampling_t4x2.ipynb`
(SR track). Nothing runs locally; everything executes on Kaggle.

## Kaggle setup (2 minutes)

1. Settings > Accelerator > GPU T4 x 2.
2. + Add Input > attach at least one sequence dataset. Suggested starter:
   `chenshu123/vimeo-triplet` + `wangsally/vimeo-90k-7` + `uom200647r/vid4-dataset` (held-out).
   REDS (`amithkesavmrajagiri/reds-dataset`), Sintel (`artemmmtry/mpi-sintel-dataset`) and
   FlyingChairs (`craljimenez/flyingchairs`) are picked up automatically when attached.
3. Keep Internet OFF. This notebook uses `/kaggle/input` only and installs nothing.
4. Run All. Outputs go to `/kaggle/working/outputs_fg`, checkpoints to `/kaggle/working/checkpoints_fg`.

## Datasets used (RESEARCH.md section 3.2 only, nothing else)

| Slug | Mount | Role here |
| :--- | :--- | :--- |
| `chenshu123/vimeo-triplet` | `/kaggle/input/vimeo-triplet` | primary FG supervision: (t-1, t) -> GT t-0.5 |
| `wangsally/vimeo-90k-7` | `/kaggle/input/vimeo-90k-7` | septuplet sliding triplets, 448x256 |
| `amithkesavmrajagiri/reds-dataset` | `/kaggle/input/reds-dataset` | consecutive-frame triplets from 720p sequences |
| `cookiemonsteryum/reds-video-superresolution-toy-dataset` | `/kaggle/input/reds-video-superresolution-toy-dataset` | <15 min smoke test (`quick_run = True`) |
| `artemmmtry/mpi-sintel-dataset` | `/kaggle/input/mpi-sintel-dataset` | consecutive clean/final triplets (flow files used only for EPE spot checks) |
| `craljimenez/flyingchairs` | `/kaggle/input/flyingchairs` | `_img1.ppm`/`_img2.ppm`/`_flow.flo` pairs for flow EPE wiring checks |
| `uom200647r/vid4-dataset` | `/kaggle/input/vid4-dataset` | HELD-OUT zero-shot test only, never trained on |
| `jesucristo/super-resolution-benchmarks` | `/kaggle/input/super-resolution-benchmarks` | ignored here (still images, no triplets) |

## What it does

- Auto-discovers frame triplets under `/kaggle/input` in any folder layout (Vimeo im1/im2/im3,
  septuplet im1..im7, REDS/Sintel consecutive frames) via sliding windows, grouped by folder.
- Trains NeuralFG (quarter-resolution flow + occlusion blend + residual, ~24K params, RESEARCH 6.3).
- RGB-only adaptation: public mirrors lack engine MV/depth, so the 10-channel G-buffer input trains as
  6-channel (frame t-1 + frame t). Widen back to 10 channels with G-buffer renders.
- Uses FP16 AMP, DataParallel on 2x T4, cuDNN benchmark, channels-last, optional torch.compile.
- Compares against average-blend and copy baselines every epoch; reports PSNR/SSIM, win rates, EPE.
- Visualises triplets, interpolated frames, occlusion maps, flow fields and error maps throughout.
- Benchmarks FG latency at 540p and 1080p, checks VRAM, exports ONNX.

## Run order

0 bootstrap, 1 logging, 2 config (`quick_run` flag), 3 hardware, 4 triplet discovery,
5 EDA, 6 loaders, 7 model, 8 loss, 9 optim, 10 train, 11 history, 12 validation,
13 visuals, 14 holdout + robustness, 15 benchmark, 16 export, 17 packaging.


In [ ]:
%matplotlib inline
import os
import sys
import json
import time
import math
import random
import struct
import shutil
import gc
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
print('python:', sys.version)
import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('matplotlib:', matplotlib.__version__)
import PIL
print('pillow:', PIL.__version__)
INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
OUT_DIR = WORKING / 'outputs_fg'
FIG_DIR = OUT_DIR / 'figs'
PRED_DIR = OUT_DIR / 'predictions'
CKPT_DIR = WORKING / 'checkpoints_fg'
LOG_DIR = WORKING / 'logs'
for d in [OUT_DIR, FIG_DIR, PRED_DIR, CKPT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('input root exists:', INPUT_ROOT.exists())
if INPUT_ROOT.exists():
    top = sorted([p.name for p in INPUT_ROOT.iterdir()])
    print('attached inputs:', top if top else ['NONE - attach a dataset first'])
print('cuda available:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count() if torch.cuda.is_available() else 0)


In [ ]:
import logging
LOG_FILE = str(LOG_DIR / 'train_fg.log')
logger = logging.getLogger('neuralfg')
logger.setLevel(logging.INFO)
logger.handlers = []
fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
ch = logging.StreamHandler(sys.stdout)
ch.setFormatter(fmt)
fh = logging.FileHandler(LOG_FILE)
fh.setFormatter(fmt)
logger.addHandler(ch)
logger.addHandler(fh)
logger.info('logging to console and file')
logger.info('internet OFF by design. No downloads, no installs, Kaggle datasets only.')
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    try:
        torch.use_deterministic_algorithms(False)
    except Exception as e:
        logger.warning('deterministic flag skipped: %s', e)
    logger.info('seeds set to %d, cudnn.benchmark True for max throughput', seed)
seed_everything(SEED)
warnings.filterwarnings('ignore')
env_rows = []
env_rows.append(['python', sys.version.split(' ')[0]])
env_rows.append(['torch', torch.__version__])
env_rows.append(['cuda_available', str(torch.cuda.is_available())])
env_rows.append(['cuda_devices', str(torch.cuda.device_count() if torch.cuda.is_available() else 0)])
env_rows.append(['cpu_count', str(os.cpu_count())])
try:
    env_rows.append(['cudnn', str(torch.backends.cudnn.version())])
except Exception:
    env_rows.append(['cudnn', 'unknown'])
env_df = pd.DataFrame(env_rows, columns=['key', 'value'])
print(env_df.to_string(index=False))
logger.info('environment dumped')


In [ ]:
@dataclass
class Config:
    seed: int = 42
    crop: int = 128  # train random crop (divisible by 4 for the 4x downsample path)
    val_crop: int = 256
    mid_channels: int = 32  # RESEARCH 6.3 default; ~24K params
    downsample: int = 4
    batch_per_gpu: int = 16
    accum_steps: int = 1
    epochs: int = 10
    lr: float = 5e-4
    weight_decay: float = 1e-4
    max_hours: float = 8.0
    quick_run: bool = False  # True -> <15 min smoke test
    limit_triplets: object = None
    limit_val: int = 300
    num_vis: int = 4
    use_amp: bool = True
    use_channels_last: bool = True
    use_compile: bool = True
    val_interval: int = 1
    keep_last_k: int = 2
cfg = Config()
# quick_run = True  # uncomment for a <15 min smoke test on the REDS toy / triplet data
if cfg.quick_run:
    cfg.epochs = 2
    cfg.limit_triplets = 400
    cfg.limit_val = 60
    cfg.num_vis = 3
    cfg.use_compile = False
    logger.info('QUICK RUN: 2 epochs, 400 triplets, compile off')
NGPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
NWORKERS = min(4, max(2, (os.cpu_count() or 4) - 1))
EFFECTIVE_BATCH = cfg.batch_per_gpu * max(1, NGPUS) * cfg.accum_steps
logger.info('mid %d downsample %dx crop %d gpus %d batch_per_gpu %d effective %d workers %d',
            cfg.mid_channels, cfg.downsample, cfg.crop, NGPUS, cfg.batch_per_gpu, EFFECTIVE_BATCH, NWORKERS)
conf_df = pd.DataFrame([{'param': k, 'value': str(v)} for k, v in asdict(cfg).items()])
conf_df.loc[len(conf_df)] = ['ngpus', str(NGPUS)]
conf_df.loc[len(conf_df)] = ['effective_batch', str(EFFECTIVE_BATCH)]
conf_df.loc[len(conf_df)] = ['num_workers', str(NWORKERS)]
print(conf_df.to_string(index=False))
with open(str(OUT_DIR / 'config_fg.json'), 'w') as f:
    json.dump({'cfg': asdict(cfg), 'ngpus': NGPUS, 'effective_batch': EFFECTIVE_BATCH, 'num_workers': NWORKERS}, f, indent=2)
logger.info('config saved to outputs_fg/config_fg.json')


In [ ]:
import subprocess
logger.info('hardware and GPU optimisation')
if shutil.which('nvidia-smi') is not None:
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free,driver_version', '--format=csv'], capture_output=True, text=True, timeout=30)
        logger.info('nvidia-smi output captured')
        print(out.stdout.strip()[:2000])
    except Exception as e:
        logger.warning('nvidia-smi failed: %s', e)
else:
    logger.warning('nvidia-smi not found')
if torch.cuda.is_available():
    for i in tqdm(range(torch.cuda.device_count()), desc='gpu-info'):
        p = torch.cuda.get_device_properties(i)
        logger.info('gpu %d: %s total %.1f GB mp %d', i, p.name, p.total_memory / 1e9, p.multi_processor_count)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        logger.info('TF32 allowed')
    except Exception as e:
        logger.warning('TF32 skipped: %s', e)
    try:
        torch.set_float32_matmul_precision('high')
        logger.info('matmul precision high')
    except Exception as e:
        logger.warning('matmul precision skipped: %s', e)
    torch.set_num_threads(os.cpu_count() or 4)
    logger.info('torch threads %d', torch.get_num_threads())
else:
    logger.warning('no CUDA device. Enable GPU T4 x 2 for full speed.')
amp_ok = torch.cuda.is_available()
compile_ok = hasattr(torch, 'compile')
hw_df = pd.DataFrame([{'check': 'GPUs', 'value': NGPUS}, {'check': 'AMP FP16', 'value': 'on' if (amp_ok and cfg.use_amp) else 'off'}, {'check': 'cuDNN benchmark', 'value': 'on'}, {'check': 'TF32', 'value': 'allowed'}, {'check': 'channels-last', 'value': str(cfg.use_channels_last)}, {'check': 'torch.compile', 'value': str(compile_ok and cfg.use_compile)}, {'check': 'DataParallel', 'value': 'yes' if NGPUS >= 2 else 'no'}, {'check': 'workers', 'value': NWORKERS}])
print(hw_df.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3))
if torch.cuda.is_available():
    mems = [torch.cuda.get_device_properties(i).total_memory / 1e9 for i in range(torch.cuda.device_count())]
    ax.bar(['gpu' + str(i) for i in range(len(mems))], mems)
    ax.set_ylabel('total VRAM (GB)')
    ax.set_title('Detected GPU memory')
else:
    ax.text(0.5, 0.5, 'no CUDA device', ha='center')
    ax.set_title('Detected GPU memory')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'gpu_memory.png'), dpi=150)
plt.show()
logger.info('saved figs/gpu_memory.png')


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff', '.ppm', '.pgm'}
HOLDOUT_HINTS = ('vid4',)  # RESEARCH 3.6: Vid4 is held-out for FG, never trained on
IGNORE_HINTS = ('super-resolution-benchmark',)  # still images, no triplets here
logger.info('triplet discovery under /kaggle/input only')
if not INPUT_ROOT.exists():
    raise RuntimeError('Kaggle-only notebook: /kaggle/input missing.')
all_paths = list(INPUT_ROOT.rglob('*'))
img_files = []
for p in tqdm(all_paths, desc='scan-input'):
    try:
        if p.is_file() and p.suffix.lower() in IMG_EXTS and not any(s.startswith('.') for s in p.parts):
            img_files.append(p)
    except Exception:
        continue
img_files = sorted(img_files)
logger.info('found %d images', len(img_files))
if len(img_files) == 0:
    raise RuntimeError('No images under /kaggle/input. Attach a sequence dataset (+ Add Input).')
def top_ds(p):
    try:
        rel = p.relative_to(INPUT_ROOT)
        return rel.parts[0] if len(rel.parts) > 1 else 'root'
    except Exception:
        return 'root'
def is_heldout(p):
    s = str(p).lower()
    return any(h in s for h in HOLDOUT_HINTS)
def is_ignored(p):
    s = str(p).lower()
    return any(h in s for h in IGNORE_HINTS)
pool_files = [p for p in img_files if not is_heldout(p) and not is_ignored(p)]
holdout_files = [p for p in img_files if is_heldout(p)]
logger.info('training pool: %d | held-out Vid4: %d | ignored stills: %d', len(pool_files), len(holdout_files), len(img_files) - len(pool_files) - len(holdout_files))
# Group training frames by parent folder, then sliding triplets (a, b, c): inputs a/c, GT mid b.
CHAIRS_SUFFIXES = ('_img1.ppm', '_img2.ppm')
by_parent = defaultdict(list)
for p in pool_files:
    if p.name.endswith(CHAIRS_SUFFIXES):
        continue  # FlyingChairs still pairs: not consecutive video frames, handled below
    by_parent[str(p.parent)].append(p)
for k in by_parent:
    by_parent[k] = sorted(by_parent[k])
triplets = []  # (fa, fb, fc, parent)
skipped_singles = 0
for parent, files in tqdm(sorted(by_parent.items()), desc='build-triplets'):
    if len(files) < 3:
        skipped_singles += 1
        continue
    for i in range(len(files) - 2):
        triplets.append((files[i], files[i + 1], files[i + 2], parent))
logger.info('triplet groups: %d dirs, %d skipped (<3 frames), %d triplets', len(by_parent), skipped_singles, len(triplets))
if len(triplets) == 0:
    raise RuntimeError('No frame triplets found. Attach vimeo-triplet, vimeo-90k-7, reds-dataset or sintel.')
# FlyingChairs pairs for flow EPE spot checks: *_img1.ppm + *_img2.ppm [+ *_flow.flo]
chairs_index = {}
for p in tqdm(pool_files, desc='index-chairs'):
    n = p.name
    if n.endswith('_img1.ppm'):
        chairs_index.setdefault((str(p.parent), n[:-9]), [None, None])[0] = p
    elif n.endswith('_img2.ppm'):
        chairs_index.setdefault((str(p.parent), n[:-9]), [None, None])[1] = p
chairs_pairs = [(a, b, None) for (a, b) in chairs_index.values() if a is not None and b is not None]
# .flo files use extension .flo (not in IMG_EXTS) so probe the expected sidecar directly
flo_count = 0
for (a, b, f) in chairs_pairs[:200]:
    cand = a.parent / (a.name.replace('_img1.ppm', '_flow.flo'))
    if cand.exists():
        flo_count += 1
logger.info('FlyingChairs pairs: %d (with .flo next to pair: %d)', len(chairs_pairs), flo_count)
counts = Counter([top_ds(t[0]) for t in triplets])
ds_df = pd.DataFrame([{'dataset': k, 'triplets': int(v)} for k, v in counts.most_common()])
ds_df['share_pct'] = (100.0 * ds_df['triplets'] / max(1, len(triplets))).round(2)
print(ds_df.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, max(2.5, 0.45 * len(ds_df))))
ax.barh(ds_df['dataset'], ds_df['triplets'])
ax.set_xlabel('triplets')
ax.set_title('FG triplets per attached Kaggle dataset')
for i, v in enumerate(ds_df['triplets']):
    ax.text(v, i, ' ' + str(v), va='center', fontsize=8)
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'dataset_counts.png'), dpi=150)
plt.show()
ds_df.to_csv(str(OUT_DIR / 'discovery_fg.csv'), index=False)
hold_df = pd.DataFrame([{'dataset': top_ds(p)} for p in holdout_files])
if len(hold_df):
    print('held-out Vid4 inventory (eval only):')
    print(hold_df.groupby('dataset').size().to_string())
logger.info('saved outputs_fg/discovery_fg.csv')


In [ ]:
logger.info('EDA: triplet resolution, motion proxy, previews')
rng = random.Random(SEED)
NS = min(600, len(triplets))
eda_idx = rng.sample(range(len(triplets)), NS)
rows = []
for i in tqdm(eda_idx, desc='eda-sizes'):
    fa, fb, fc, parent = triplets[i]
    try:
        with Image.open(fa) as im:
            im.load()
            w, h, mode = im.size[0], im.size[1], im.mode
        with Image.open(fb) as imb:
            imb.load()
            a = np.asarray(Image.open(fa).convert('RGB').resize((64, 64)), dtype=np.float32) / 255.0
            b = np.asarray(imb.convert('RGB').resize((64, 64)), dtype=np.float32) / 255.0
            motion = float(np.abs(a - b).mean())
        rows.append([w, h, mode, motion, top_ds(fa)])
    except Exception as e:
        logger.warning('skip corrupt triplet: %s', e)
edadf = pd.DataFrame(rows, columns=['W', 'H', 'mode', 'motion', 'dataset'])
edadf['mp'] = edadf['W'] * edadf['H'] / 1e6
print(edadf[['W', 'H', 'mp', 'motion']].describe().round(3).to_string())
print(edadf['mode'].value_counts().head(8).to_string())
print(edadf['dataset'].value_counts().head(8).to_string())
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].hist(edadf['W'], bins=25)
axes[0, 0].set_title('width')
axes[0, 1].hist(edadf['H'], bins=25)
axes[0, 1].set_title('height')
axes[1, 0].hist(edadf['motion'], bins=30)
axes[1, 0].set_title('motion proxy |t - mid| (64px thumb)')
axes[1, 1].hist(edadf['mp'], bins=25)
axes[1, 1].set_title('megapixels')
plt.suptitle('FG triplet EDA')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'eda_triplets.png'), dpi=150)
plt.show()
# Preview 4 triplets: f(t-1), GT mid, f(t), abs motion
fig, axes = plt.subplots(4, 4, figsize=(14, 12))
for r in range(4):
    fa, fb, fc, parent = triplets[rng.randrange(len(triplets))]
    try:
        ia = np.asarray(Image.open(fa).convert('RGB').resize((224, 128)))
        ib = np.asarray(Image.open(fb).convert('RGB').resize((224, 128)))
        ic = np.asarray(Image.open(fc).convert('RGB').resize((224, 128)))
    except Exception:
        continue
    mot = np.abs(ia.astype(np.float32) - ic.astype(np.float32)).mean(axis=2) / 255.0
    axes[r, 0].imshow(ia)
    axes[r, 0].set_title('t-1', fontsize=9)
    axes[r, 0].axis('off')
    axes[r, 1].imshow(ib)
    axes[r, 1].set_title('GT mid', fontsize=9)
    axes[r, 1].axis('off')
    axes[r, 2].imshow(ic)
    axes[r, 2].set_title('t', fontsize=9)
    axes[r, 2].axis('off')
    axes[r, 3].imshow(mot, vmin=0, vmax=0.5)
    axes[r, 3].set_title('|t-1 - t| motion', fontsize=9)
    axes[r, 3].axis('off')
plt.suptitle('Triplet previews (inputs t-1/t, GT mid, motion)')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'triplet_preview.png'), dpi=150)
plt.show()
logger.info('saved eda_triplets.png and triplet_preview.png')


In [ ]:
import torch.nn.functional as F
import torchvision.transforms.functional as TF
logger.info('FG paired triplet loading (inputs t-1/t, GT mid)')
class TripletFGDataset(torch.utils.data.Dataset):
    'Sliding triplets: model sees (t-1, t), supervised by GT mid frame.'
    def __init__(self, trips, crop=128, is_train=True, seed=42):
        self.trips = list(trips)
        self.crop = crop
        self.is_train = is_train
        self.rng = random.Random(seed + (0 if is_train else 7777))
    def __len__(self):
        return len(self.trips)
    def _open(self, p):
        with Image.open(p) as im:
            im.load()
            return im.convert('RGB')
    def __getitem__(self, idx):
        fa, fb, fc, _parent = self.trips[idx % len(self.trips)]
        try:
            ia, ib, ic = self._open(fa), self._open(fb), self._open(fc)
        except Exception:
            ia = ib = ic = self._open(self.trips[(idx + 1) % len(self.trips)][1])
        w, h = ib.size
        cw = min(self.crop, w - (w % 4))
        ch = min(self.crop, h - (h % 4))
        cw = max(cw - (cw % 4), 32)
        ch = max(ch - (ch % 4), 32)
        if self.is_train:
            x = self.rng.randint(0, max(0, w - cw))
            y = self.rng.randint(0, max(0, h - ch))
            flip = self.rng.random() < 0.5
        else:
            x, y, flip = (w - cw) // 2, (h - ch) // 2, False
        box = (x, y, x + cw, y + ch)
        ia, ib, ic = ia.crop(box), ib.crop(box), ic.crop(box)
        if flip:
            ia, ib, ic = ia.transpose(Image.FLIP_LEFT_RIGHT), ib.transpose(Image.FLIP_LEFT_RIGHT), ic.transpose(Image.FLIP_LEFT_RIGHT)
        return TF.to_tensor(ia), TF.to_tensor(ib), TF.to_tensor(ic)
# Dir-level split: whole folders go to one split (no sequence leakage).
parents = sorted(set(t[3] for t in triplets))
rngs = random.Random(SEED)
rngs.shuffle(parents)
n_val_p = max(1, int(0.10 * len(parents)))
n_test_p = max(1, int(0.10 * len(parents)))
if len(parents) < 10:
    n_val_p = max(1, len(parents) // 5)
    n_test_p = max(1, len(parents) // 5)
val_par, test_par = set(parents[len(parents) - n_val_p - n_test_p:len(parents) - n_test_p]), set(parents[len(parents) - n_test_p:])
train_trips = [t for t in triplets if t[3] not in val_par and t[3] not in test_par]
val_trips = [t for t in triplets if t[3] in val_par]
test_trips = [t for t in triplets if t[3] in test_par]
if cfg.limit_triplets is not None:
    train_trips = train_trips[:int(cfg.limit_triplets)]
val_trips = val_trips[:int(cfg.limit_val)]
test_trips = test_trips[:int(cfg.limit_val)]
print(pd.DataFrame([{'split': 'train', 'triplets': len(train_trips)}, {'split': 'val', 'triplets': len(val_trips)}, {'split': 'test', 'triplets': len(test_trips)}]).to_string(index=False))
logger.info('dir-level split over %d parent folders', len(parents))
train_ds = TripletFGDataset(train_trips, crop=cfg.crop, is_train=True, seed=SEED)
val_ds = TripletFGDataset(val_trips, crop=cfg.val_crop, is_train=False, seed=SEED)
test_ds = TripletFGDataset(test_trips, crop=cfg.val_crop, is_train=False, seed=SEED)
loader_kw = {'num_workers': NWORKERS, 'pin_memory': True, 'drop_last': True} if NWORKERS > 0 else {'num_workers': 0, 'pin_memory': False, 'drop_last': True}
if NWORKERS > 0:
    loader_kw['persistent_workers'] = True
    loader_kw['prefetch_factor'] = 4
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=cfg.batch_per_gpu * max(1, NGPUS), shuffle=True, **loader_kw)
val_kw = dict(loader_kw)
val_kw['drop_last'] = False
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=cfg.batch_per_gpu * max(1, NGPUS), shuffle=False, **val_kw)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=cfg.batch_per_gpu * max(1, NGPUS), shuffle=False, **val_kw)
logger.info('train batches %d val %d test %d', len(train_loader), len(val_loader), len(test_loader))
f0_b, fm_b, f1_b = next(iter(train_loader))
logger.info('batch shapes f0 %s mid %s f1 %s', tuple(f0_b.shape), tuple(fm_b.shape), tuple(f1_b.shape))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(min(4, f0_b.shape[0])):
    axes[0, i].imshow(f0_b[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title('input t-1', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(fm_b[i].permute(1, 2, 0).numpy())
    axes[1, i].set_title('GT mid', fontsize=9)
    axes[1, i].axis('off')
plt.suptitle('FG batch preview')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'batch_preview_fg.png'), dpi=150)
plt.show()
t0 = time.time()
NB = min(20, len(train_loader))
it = iter(train_loader)
for _ in tqdm(range(NB), desc='loader-benchmark'):
    try:
        b = next(it)
    except StopIteration:
        break
dt = time.time() - t0
ips = (NB * f0_b.shape[0] * 3) / max(1e-6, dt)
logger.info('loader %d batches in %.2f s: %.1f frames per s', NB, dt, ips)
print(pd.DataFrame([{'metric': 'frames_per_s', 'value': round(ips, 1)}, {'metric': 'ms_per_batch', 'value': round(1000.0 * dt / max(1, NB), 1)}]).to_string(index=False))


In [ ]:
import torch.nn as nn
logger.info('model: NeuralFG quarter-resolution flow + occlusion blend (RESEARCH 6.3, RGB 6ch adaptation)')
class NeuralFG(nn.Module):
    'Compact learned intermediate flow estimator. G-buffer build uses in_ch=10; public RGB uses 6.'
    def __init__(self, in_channels=6, mid_channels=32, downsample=4):
        super().__init__()
        self.downsample = downsample
        m = mid_channels
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, m, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(m, m, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(m, m, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(m, 9, 3, padding=1),
        )
    def forward(self, f_prev, f_curr):
        B, _, H, W = f_prev.shape
        x = torch.cat([f_prev, f_curr], dim=1)
        x_ds = F.avg_pool2d(x, self.downsample)
        pred = self.encoder(x_ds)
        pred = F.interpolate(pred, size=(H, W), mode='bilinear', align_corners=False)
        flow_0 = pred[:, 0:2]
        flow_1 = pred[:, 2:4]
        occ = torch.sigmoid(pred[:, 4:6])
        residual = pred[:, 6:9]
        gy, gx = torch.meshgrid(torch.linspace(-1, 1, H, device=f_prev.device), torch.linspace(-1, 1, W, device=f_prev.device), indexing='ij')
        grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).expand(B, H, W, 2)
        def _warp(im, fl):
            fn = torch.stack([2.0 * fl[:, 0] / max(1, W), 2.0 * fl[:, 1] / max(1, H)], dim=-1)
            return F.grid_sample(im, grid + fn, mode='bilinear', padding_mode='border', align_corners=False)
        w0 = _warp(f_prev, flow_0)
        w1 = _warp(f_curr, flow_1)
        return (occ[:, 0:1] * w0 + occ[:, 1:2] * w1 + residual).clamp(0, 1), {'flow_0': flow_0, 'flow_1': flow_1, 'occ': occ, 'residual': residual}
    def infer(self, f_prev, f_curr):
        out, _ = self.forward(f_prev, f_curr)
        return out
def count_params(m):
    return sum(p.numel() for p in m.parameters())
model = NeuralFG(in_channels=6, mid_channels=cfg.mid_channels, downsample=cfg.downsample)
logger.info('NeuralFG params %d (budget <=200K: %s)', count_params(model), count_params(model) <= 200000)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
if cfg.use_channels_last and device.type == 'cuda':
    model = model.to(memory_format=torch.channels_last)
    logger.info('channels-last on')
if cfg.use_compile and hasattr(torch, 'compile') and device.type == 'cuda':
    try:
        model = torch.compile(model, mode='default')
        logger.info('torch.compile on (default mode for fast first run)')
    except Exception as e:
        logger.warning('compile skipped: %s', e)
if NGPUS >= 2:
    model = nn.DataParallel(model)
    logger.info('DataParallel over %d GPUs', NGPUS)
model.train()
with torch.no_grad():
    d0 = torch.randn(2, 3, 64, 64, device=device)
    d1 = torch.randn(2, 3, 64, 64, device=device)
    if cfg.use_channels_last and device.type == 'cuda':
        d0 = d0.to(memory_format=torch.channels_last)
        d1 = d1.to(memory_format=torch.channels_last)
    y, aux = model(d0, d1)
    logger.info('forward check passed %s occ-mean %.3f', tuple(y.shape), float(aux['occ'].mean()))
    assert y.shape == d0.shape
print(pd.DataFrame([{'item': 'params', 'value': count_params(model)}, {'item': 'budget', 'value': 200000}, {'item': 'within_budget', 'value': str(count_params(model) <= 200000)}, {'item': 'downsample', 'value': str(cfg.downsample) + 'x'}]).to_string(index=False))


In [ ]:
logger.info('losses and metrics from scratch, offline safe')
class Charbonnier(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, a, b):
        return torch.sqrt((a - b) ** 2 + self.eps ** 2).mean()
def rgb_to_y(x):
    return 0.25 * x[:, 0:1] + 0.5 * x[:, 1:2] + 0.25 * x[:, 2:3]
def census_code(y, k=10.0):
    # y: [B,1,H,W] -> soft 3x3 census vs center, sigmoid Hamming-friendly
    patches = F.unfold(y, kernel_size=3, padding=1)  # [B,9,H*W]
    center = patches[:, 4:5, :]
    return torch.sigmoid((patches - center) * k)
def census_loss(pred, gt, k=10.0):
    cy_p = census_code(rgb_to_y(pred), k)
    cy_g = census_code(rgb_to_y(gt), k)
    return (cy_p - cy_g).abs().mean()
KX = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
KY = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
def edge_loss(a, b):
    w = a.shape[1]
    kx = KX.repeat(w, 1, 1, 1).to(a.device).type_as(a)
    ky = KY.repeat(w, 1, 1, 1).to(a.device).type_as(a)
    ax = F.conv2d(a, kx, padding=1, groups=w)
    ay = F.conv2d(a, ky, padding=1, groups=w)
    bx = F.conv2d(b, kx, padding=1, groups=w)
    by = F.conv2d(b, ky, padding=1, groups=w)
    return (ax - bx).abs().mean() + (ay - by).abs().mean()
class FGLoss(nn.Module):
    def __init__(self, w_char=1.0, w_census=0.5, w_edge=0.2):
        super().__init__()
        self.char = Charbonnier()
        self.w_char = w_char
        self.w_census = w_census
        self.w_edge = w_edge
    def forward(self, pred, gt):
        lc = self.char(pred, gt)
        lx = census_loss(pred, gt)
        le = edge_loss(pred, gt)
        return self.w_char * lc + self.w_census * lx + self.w_edge * le, {'char': float(lc.detach()), 'census': float(lx.detach()), 'edge': float(le.detach())}
def psnr_batch(a, b):
    mse = ((a - b) ** 2).mean(dim=(1, 2, 3))
    return 10 * torch.log10(1.0 / (mse + 1e-8))
def _gauss(ch, win=11, sigma=1.5):
    c = torch.arange(win).float() - win // 2
    g = torch.exp(-(c ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w = g[:, None] * g[None, :]
    return w.expand(ch, 1, win, win).contiguous()
def ssim_batch(a, b, win=11):
    ch = a.shape[1]
    w = _gauss(ch, win).to(a.device).type_as(a)
    pad = win // 2
    m1 = F.conv2d(a, w, padding=pad, groups=ch)
    m2 = F.conv2d(b, w, padding=pad, groups=ch)
    v1 = F.conv2d(a * a, w, padding=pad, groups=ch) - m1 * m1
    v2 = F.conv2d(b * b, w, padding=pad, groups=ch) - m2 * m2
    v12 = F.conv2d(a * b, w, padding=pad, groups=ch) - m1 * m2
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    m = (2 * m1 * m2 + c1) * (2 * v12 + c2) / (((m1 * m1 + m2 * m2 + c1) * (v1 + v2 + c2)) + 1e-8)
    return m.mean(dim=(1, 2, 3))
def epe_batch(flow_pred, flow_gt):
    # flow_*: [B,2,H,W] in pixels; mean end-point error
    return torch.sqrt(((flow_pred - flow_gt) ** 2).sum(dim=1) + 1e-12).mean(dim=(1, 2))
def read_flo(path):
    # Middlebury .flo, little-endian: magic 202021.25, w int32, h int32, interleaved float32
    with open(str(path), 'rb') as f:
        magic = struct.unpack('<f', f.read(4))[0]
        if abs(magic - 202021.25) > 0.5:
            raise ValueError('bad .flo magic %s in %s' % (magic, path))
        w = struct.unpack('<i', f.read(4))[0]
        h = struct.unpack('<i', f.read(4))[0]
        data = np.fromfile(f, dtype=np.float32, count=2 * h * w)
    if data.size != 2 * h * w:
        raise ValueError('truncated .flo %s' % path)
    return data.reshape(h, w, 2).transpose(2, 0, 1)  # [2,H,W]
criterion = FGLoss(w_char=1.0, w_census=0.5, w_edge=0.2)
logger.info('loss weights char 1.0 census 0.5 edge 0.2 (RESEARCH 7.2 adapted, LPIPS/VGG skipped offline)')
with torch.no_grad():
    ta = torch.rand(2, 3, 64, 64)
    tb = torch.rand(2, 3, 64, 64)
    tot, parts = criterion(ta, tb)
    print(pd.DataFrame([{'metric': 'charbonnier', 'value': round(parts['char'], 5)}, {'metric': 'census', 'value': round(parts['census'], 5)}, {'metric': 'edge', 'value': round(parts['edge'], 5)}, {'metric': 'total', 'value': round(float(tot), 5)}, {'metric': 'psnr_random', 'value': round(float(psnr_batch(ta, tb).mean()), 2)}, {'metric': 'ssim_random', 'value': round(float(ssim_batch(ta, tb).mean()), 4)}]).to_string(index=False))
    logger.info('loss sanity check passed')


In [ ]:
logger.info('optimiser, scheduler, AMP, EMA, checkpoints')
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.999), weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)
USE_AMP = bool(device.type == 'cuda' and cfg.use_amp)
scaler = None
if USE_AMP:
    try:
        scaler = torch.amp.GradScaler('cuda')
        logger.info('GradScaler on')
    except Exception as e:
        logger.warning('scaler fallback: %s', e)
        scaler = torch.cuda.amp.GradScaler()
else:
    logger.info('AMP off, fp32 training')
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone()
    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)
    @torch.no_grad()
    def copy_to(self, model):
        bak = {}
        for n, p in model.named_parameters():
            if n in self.shadow:
                bak[n] = p.detach().clone()
                p.copy_(self.shadow[n])
        return bak
    @torch.no_grad()
    def restore(self, model, bak):
        for n, p in model.named_parameters():
            if n in bak:
                p.copy_(bak[n])
    def state_dict(self):
        return {'decay': self.decay, 'shadow': self.shadow}
    def load_state_dict(self, sd):
        self.decay = sd.get('decay', 0.999)
        self.shadow = sd['shadow']
ema = EMA(model, decay=0.999)
logger.info('EMA decay 0.999')
def save_ckpt(path, model, opt, sch, scl, ema_o, epoch, step, best):
    d = {'epoch': epoch, 'step': step, 'best_psnr': best, 'model': model.state_dict(), 'optim': opt.state_dict(), 'sched': sch.state_dict(), 'ema': ema_o.state_dict(), 'cfg': asdict(cfg)}
    if scl is not None:
        d['scaler'] = scl.state_dict()
    d['rng'] = {'torch': torch.get_rng_state(), 'numpy': np.random.get_state(), 'python': random.getstate()}
    torch.save(d, path)
    logger.info('saved %s epoch %d best %.2f', path, epoch, best)
def load_ckpt(path, model, opt, sch, scl, ema_o):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['optim'])
    sch.load_state_dict(ck['sched'])
    if scl is not None and 'scaler' in ck:
        scl.load_state_dict(ck['scaler'])
    ema_o.load_state_dict(ck['ema'])
    logger.info('resumed %s epoch %d best %.2f', path, ck['epoch'], ck['best_psnr'])
    return ck['epoch'], ck['step'], ck['best_psnr']
start_epoch, gstep, best_psnr = 0, 0, -1.0
LAST_CKPT = CKPT_DIR / 'last_fg.pt'
BEST_CKPT = CKPT_DIR / 'best_fg.pt'
if LAST_CKPT.exists():
    try:
        start_epoch, gstep, best_psnr = load_ckpt(str(LAST_CKPT), model, optimizer, scheduler, scaler, ema)
        start_epoch = start_epoch + 1
    except Exception as e:
        logger.warning('resume failed, fresh start: %s', e)
print(pd.DataFrame([{'item': 'lr', 'value': cfg.lr}, {'item': 'weight_decay', 'value': cfg.weight_decay}, {'item': 'scheduler', 'value': 'cosine'}, {'item': 'amp', 'value': str(USE_AMP)}, {'item': 'start_epoch', 'value': start_epoch}, {'item': 'best_psnr', 'value': round(best_psnr, 2)}]).to_string(index=False))


In [ ]:
logger.info('training NeuralFG with AMP, baselines tracked')
HIST_CSV = OUT_DIR / 'history_fg.csv'
T_START = time.time()
BUDGET_S = cfg.max_hours * 3600.0
def _fwd(f0, f1):
    if cfg.use_channels_last and device.type == 'cuda':
        f0 = f0.to(memory_format=torch.channels_last)
    if USE_AMP:
        try:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                return model(f0, f1)
        except Exception:
            with torch.cuda.amp.autocast():
                return model(f0, f1)
    return model(f0, f1)
def run_validate(loader, use_ema=True):
    was_train = model.training
    model.eval()
    bak = ema.copy_to(model) if use_ema else None
    tot, n, ps, ss, ps_avg, ps_copy = 0.0, 0, 0.0, 0.0, 0.0, 0.0
    with torch.no_grad():
        for f0, fm, f1 in tqdm(loader, desc='validate', leave=False):
            f0 = f0.to(device, non_blocking=True)
            fm = fm.to(device, non_blocking=True)
            f1 = f1.to(device, non_blocking=True)
            out = _fwd(f0, f1)
            pred = out[0] if isinstance(out, (tuple, list)) else out
            pred = pred.clamp(0, 1)
            avg = ((f0 + f1) / 2).clamp(0, 1)
            loss, _ = criterion(pred, fm)
            tot += float(loss) * f0.shape[0]
            ps += float(psnr_batch(pred, fm).mean()) * f0.shape[0]
            ss += float(ssim_batch(pred, fm).mean()) * f0.shape[0]
            ps_avg += float(psnr_batch(avg, fm).mean()) * f0.shape[0]
            ps_copy += float(psnr_batch(f0, fm).mean()) * f0.shape[0]
            n += f0.shape[0]
    if use_ema:
        ema.restore(model, bak)
    if was_train:
        model.train()
    return tot / max(1, n), ps / max(1, n), ss / max(1, n), ps_avg / max(1, n), ps_copy / max(1, n)
history = []
if HIST_CSV.exists() and start_epoch > 0:
    try:
        history = pd.read_csv(str(HIST_CSV)).to_dict(orient='records')
        logger.info('loaded history rows %d', len(history))
    except Exception:
        history = []
for epoch in tqdm(range(start_epoch, cfg.epochs), desc='epochs'):
    if time.time() - T_START > BUDGET_S - 600:
        logger.warning('time budget guard: stopping to allow packaging')
        break
    model.train()
    run_loss, run_psnr, seen = 0.0, 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc='train e%d' % epoch, leave=False)
    for bi, (f0, fm, f1) in enumerate(pbar):
        f0 = f0.to(device, non_blocking=True)
        fm = fm.to(device, non_blocking=True)
        f1 = f1.to(device, non_blocking=True)
        if USE_AMP:
            try:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    out = model(f0, f1)
                    pred = out[0] if isinstance(out, (tuple, list)) else out
                    loss_raw, _ = criterion(pred, fm)
                    loss = loss_raw / cfg.accum_steps
            except Exception:
                with torch.cuda.amp.autocast():
                    out = model(f0, f1)
                    pred = out[0] if isinstance(out, (tuple, list)) else out
                    loss_raw, _ = criterion(pred, fm)
                    loss = loss_raw / cfg.accum_steps
            scaler.scale(loss).backward()
            if (bi + 1) % cfg.accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
        else:
            out = model(f0, f1)
            pred = out[0] if isinstance(out, (tuple, list)) else out
            loss_raw, _ = criterion(pred, fm)
            loss = loss_raw / cfg.accum_steps
            loss.backward()
            if (bi + 1) % cfg.accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
        gstep = gstep + 1
        with torch.no_grad():
            b_psnr = float(psnr_batch(pred.detach().clamp(0, 1), fm).mean())
        run_loss += float(loss_raw.detach())
        run_psnr += b_psnr
        seen += 1
        pbar.set_postfix({'loss': '%.4f' % (run_loss / seen), 'psnr': '%.2f' % (run_psnr / seen), 'lr': '%.2e' % optimizer.param_groups[0]['lr']})
    scheduler.step()
    tr_loss, tr_psnr = run_loss / max(1, seen), run_psnr / max(1, seen)
    if (epoch + 1) % cfg.val_interval == 0:
        va_loss, va_psnr, va_ssim, va_avg, va_copy = run_validate(val_loader, use_ema=True)
    else:
        va_loss, va_psnr, va_ssim, va_avg, va_copy = float('nan'), float('nan'), float('nan'), float('nan'), float('nan')
    row = {'epoch': epoch, 'train_loss': round(tr_loss, 5), 'train_psnr': round(tr_psnr, 3), 'val_loss': round(float(va_loss), 5), 'val_psnr': round(float(va_psnr), 3), 'val_ssim': round(float(va_ssim), 5), 'val_avg_psnr': round(float(va_avg), 3), 'val_copy_psnr': round(float(va_copy), 3), 'lr': optimizer.param_groups[0]['lr'], 'best': round(best_psnr, 3)}
    history.append(row)
    pd.DataFrame(history).to_csv(str(HIST_CSV), index=False)
    logger.info('epoch %d train %.4f psnr %.2f | val %.4f psnr %.2f ssim %.4f avg %.2f copy %.2f', epoch, tr_loss, tr_psnr, va_loss, va_psnr, va_ssim, va_avg, va_copy)
    save_ckpt(str(LAST_CKPT), model, optimizer, scheduler, scaler, ema, epoch, gstep, best_psnr)
    if va_psnr == va_psnr and va_psnr > best_psnr:
        best_psnr = float(va_psnr)
        save_ckpt(str(BEST_CKPT), model, optimizer, scheduler, scaler, ema, epoch, gstep, best_psnr)
        logger.info('new best %.3f', best_psnr)
    ep_ckpt = CKPT_DIR / ('epoch_fg_%03d.pt' % epoch)
    save_ckpt(str(ep_ckpt), model, optimizer, scheduler, scaler, ema, epoch, gstep, best_psnr)
    olds = sorted(CKPT_DIR.glob('epoch_fg_*.pt'))
    for old in olds[:-cfg.keep_last_k]:
        try:
            old.unlink()
        except Exception:
            pass
print(pd.DataFrame(history).to_string(index=False))
logger.info('training done. best val psnr %.3f', best_psnr)


In [ ]:
logger.info('FG training history plots and table')
hist_df = pd.read_csv(str(HIST_CSV))
print(hist_df.to_string(index=False))
best_row = hist_df.loc[hist_df['val_psnr'].idxmax()]
logger.info('best epoch %d val psnr %.3f ssim %.5f (avg baseline %.3f, copy %.3f)', int(best_row['epoch']), float(best_row['val_psnr']), float(best_row['val_ssim']), float(best_row['val_avg_psnr']), float(best_row['val_copy_psnr']))
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(hist_df['epoch'], hist_df['train_loss'], marker='o', label='train')
axes[0, 0].plot(hist_df['epoch'], hist_df['val_loss'], marker='o', label='val')
axes[0, 0].set_title('FG loss')
axes[0, 0].legend()
axes[0, 1].plot(hist_df['epoch'], hist_df['train_psnr'], marker='o', label='train')
axes[0, 1].plot(hist_df['epoch'], hist_df['val_psnr'], marker='o', label='model')
axes[0, 1].plot(hist_df['epoch'], hist_df['val_avg_psnr'], marker='x', label='avg baseline')
axes[0, 1].plot(hist_df['epoch'], hist_df['val_copy_psnr'], marker='x', label='copy baseline')
axes[0, 1].set_title('PSNR dB, higher better')
axes[0, 1].legend(fontsize=8)
axes[1, 0].plot(hist_df['epoch'], hist_df['val_ssim'], marker='o', color='green')
axes[1, 0].set_title('val SSIM, higher better')
axes[1, 1].plot(hist_df['epoch'], hist_df['lr'], marker='o', color='purple')
axes[1, 1].set_title('learning rate')
plt.suptitle('NeuralFG training curves')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'training_curves_fg.png'), dpi=150)
plt.show()
logger.info('saved figs/training_curves_fg.png')


In [ ]:
logger.info('deep validation: model vs average-blend vs copy baselines')
if BEST_CKPT.exists():
    ck = torch.load(str(BEST_CKPT), map_location=device)
    model.load_state_dict(ck['model'])
    logger.info('loaded best checkpoint epoch %d', ck['epoch'])
model.eval()
rows = []
with torch.no_grad():
    for f0, fm, f1 in tqdm(val_loader, desc='val-per-batch'):
        f0 = f0.to(device)
        fm = fm.to(device)
        f1 = f1.to(device)
        out = _fwd(f0, f1)
        pred = (out[0] if isinstance(out, (tuple, list)) else out).clamp(0, 1)
        avg = ((f0 + f1) / 2).clamp(0, 1)
        pm = psnr_batch(pred, fm).cpu()
        sm = ssim_batch(pred, fm).cpu()
        pa = psnr_batch(avg, fm).cpu()
        sa = ssim_batch(avg, fm).cpu()
        pc = psnr_batch(f0, fm).cpu()
        for i in range(f0.shape[0]):
            rows.append({'psnr_model': float(pm[i]), 'ssim_model': float(sm[i]), 'psnr_avg': float(pa[i]), 'ssim_avg': float(sa[i]), 'psnr_copy': float(pc[i]), 'gain_over_avg': float(pm[i] - pa[i]), 'gain_over_copy': float(pm[i] - pc[i])})
valm = pd.DataFrame(rows)
valm.to_csv(str(OUT_DIR / 'val_metrics_fg.csv'), index=False)
print(valm[['psnr_model', 'psnr_avg', 'psnr_copy', 'gain_over_avg']].describe().round(4).to_string())
agg = pd.DataFrame([{'metric': 'PSNR model', 'value': round(valm['psnr_model'].mean(), 3)}, {'metric': 'PSNR avg-blend', 'value': round(valm['psnr_avg'].mean(), 3)}, {'metric': 'PSNR copy', 'value': round(valm['psnr_copy'].mean(), 3)}, {'metric': 'gain over avg', 'value': round(valm['gain_over_avg'].mean(), 3)}, {'metric': 'SSIM model', 'value': round(valm['ssim_model'].mean(), 5)}, {'metric': 'share beats avg pct', 'value': round(100.0 * (valm['gain_over_avg'] > 0).mean(), 2)}])
print(agg.to_string(index=False))
logger.info('val aggregate logged')
print('top 5 by gain over avg:')
print(valm.sort_values('gain_over_avg', ascending=False).head(5).round(3).to_string(index=False))
print('bottom 5 by gain over avg:')
print(valm.sort_values('gain_over_avg').head(5).round(3).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(valm['psnr_avg'], bins=30, alpha=0.6, label='avg baseline')
axes[0].hist(valm['psnr_model'], bins=30, alpha=0.6, label='model')
axes[0].set_title('FG PSNR distribution')
axes[0].legend()
axes[1].hist(valm['gain_over_avg'], bins=30)
axes[1].set_title('PSNR gain over avg-blend')
axes[1].axvline(0, linestyle='--')
axes[2].scatter(valm['psnr_avg'], valm['psnr_model'], s=8, alpha=0.5)
axes[2].set_xlabel('avg baseline PSNR')
axes[2].set_ylabel('model PSNR')
axes[2].set_title('model vs avg-blend')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'val_analysis_fg.png'), dpi=150)
plt.show()
logger.info('saved val_metrics_fg.csv and val_analysis_fg.png')


In [ ]:
logger.info('visual FG predictions: frames, occlusion, flow, error maps')
model.eval()
NV = min(cfg.num_vis, len(val_trips))
pick_idx = random.Random(SEED).sample(range(len(val_trips)), NV)
fig, axes = plt.subplots(NV, 6, figsize=(18, 3.4 * NV))
if NV == 1:
    axes = np.asarray([axes])
table_rows = []
for r, ti in enumerate(tqdm(pick_idx, desc='visuals')):
    f0_t, fm_t, f1_t = val_ds[ti]
    f0_b = f0_t.unsqueeze(0).to(device)
    f1_b = f1_t.unsqueeze(0).to(device)
    fm_b = fm_t.unsqueeze(0).to(device)
    with torch.no_grad():
        out = _fwd(f0_b, f1_b)
        if isinstance(out, (tuple, list)):
            pred_b, aux = out[0].clamp(0, 1), out[1]
        else:
            pred_b, aux = out.clamp(0, 1), None
    avg_b = ((f0_b + f1_b) / 2).clamp(0, 1)
    pm = float(psnr_batch(pred_b, fm_b).mean())
    pa = float(psnr_batch(avg_b, fm_b).mean())
    table_rows.append({'triplet': ti, 'psnr_model': round(pm, 2), 'psnr_avg': round(pa, 2), 'gain': round(pm - pa, 2)})
    f0_np = f0_b[0].cpu().permute(1, 2, 0).numpy()
    f1_np = f1_b[0].cpu().permute(1, 2, 0).numpy()
    fm_np = fm_b[0].cpu().permute(1, 2, 0).numpy()
    pr_np = pred_b[0].cpu().permute(1, 2, 0).numpy()
    err = np.abs(pr_np - fm_np).mean(axis=2)
    axes[r, 0].imshow(f0_np)
    axes[r, 0].set_title('input t-1', fontsize=9)
    axes[r, 0].axis('off')
    axes[r, 1].imshow(f1_np)
    axes[r, 1].set_title('input t', fontsize=9)
    axes[r, 1].axis('off')
    axes[r, 2].imshow(pr_np)
    axes[r, 2].set_title('model mid %.2f dB' % pm, fontsize=9)
    axes[r, 2].axis('off')
    axes[r, 3].imshow(fm_np)
    axes[r, 3].set_title('GT mid', fontsize=9)
    axes[r, 3].axis('off')
    axes[r, 4].imshow(err, vmin=0, vmax=0.25)
    axes[r, 4].set_title('abs error', fontsize=9)
    axes[r, 4].axis('off')
    if aux is not None and 'occ' in aux:
        occ0 = aux['occ'][0, 0].detach().float().cpu().numpy()
        axes[r, 5].imshow(occ0, vmin=0, vmax=1)
        axes[r, 5].set_title('occlusion w0', fontsize=9)
        axes[r, 5].axis('off')
    else:
        axes[r, 5].imshow(np.abs(pr_np - ((f0_np + f1_np) / 2)).mean(axis=2), vmin=0, vmax=0.25)
        axes[r, 5].set_title('diff vs avg', fontsize=9)
        axes[r, 5].axis('off')
    Image.fromarray((pr_np * 255).astype(np.uint8)).save(str(PRED_DIR / ('fg_%d.png' % r)))
plt.suptitle('FG: inputs vs model mid vs GT mid + error + occlusion')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'predictions_grid_fg.png'), dpi=150)
plt.show()
vis_df = pd.DataFrame(table_rows)
print(vis_df.to_string(index=False))
# Flow quiver for first visual triplet
try:
    f0_t, fm_t, f1_t = val_ds[pick_idx[0]]
    with torch.no_grad():
        out = _fwd(f0_t.unsqueeze(0).to(device), f1_t.unsqueeze(0).to(device))
        aux = out[1] if isinstance(out, (tuple, list)) else None
    if aux is not None:
        fl = aux['flow_0'][0].detach().float().cpu().numpy()  # [2,H,W] pixels
        bg = f0_t.permute(1, 2, 0).numpy()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.imshow(bg)
        H, W = fl.shape[1], fl.shape[2]
        step = max(8, H // 24)
        yy, xx = np.mgrid[0:H:step, 0:W:step]
        ax.quiver(xx, yy, fl[0, ::step, ::step], fl[1, ::step, ::step], color='cyan', scale=120, width=0.004)
        ax.set_title('estimated flow t-1 -> mid (quiver over input)')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(str(FIG_DIR / 'flow_quiver.png'), dpi=150)
        plt.show()
        mag = np.sqrt((fl ** 2).sum(axis=0))
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(mag.ravel(), bins=40)
        ax.set_title('flow magnitude histogram (px)')
        plt.tight_layout()
        plt.savefig(str(FIG_DIR / 'flow_magnitude.png'), dpi=150)
        plt.show()
except Exception as e:
    logger.warning('flow quiver skipped: %s', e)
logger.info('saved predictions_grid_fg.png, flow_quiver.png and %d fg pngs', NV)


In [ ]:
logger.info('holdout test + robustness (scene-cut, disocclusion, FlyingChairs EPE)')
test_loss, test_psnr, test_ssim, test_avg, test_copy = run_validate(test_loader, use_ema=True)
print(pd.DataFrame([{'split': 'val best', 'psnr': round(float(best_row['val_psnr']), 3), 'ssim': round(float(best_row['val_ssim']), 5)}, {'split': 'test', 'psnr': round(test_psnr, 3), 'ssim': round(test_ssim, 5)}, {'split': 'test avg-blend', 'psnr': round(test_avg, 3), 'ssim': None}]).to_string(index=False))
logger.info('test psnr %.3f ssim %.5f (avg baseline %.3f)', test_psnr, test_ssim, test_avg)
# Scene-cut bypass demo: mismatched pair should fall back toward copy, not smear.
model.eval()
with torch.no_grad():
    f0_t, fm_t, f1_t = val_ds[0]
    f0_b = f0_t.unsqueeze(0).to(device)
    f1_rand = torch.rand_like(f0_b)
    out_bad = _fwd(f0_b, f1_rand)
    pred_bad = (out_bad[0] if isinstance(out_bad, (tuple, list)) else out_bad).clamp(0, 1)
    bad_np = pred_bad[0].cpu().permute(1, 2, 0).numpy()
    f0_np = f0_b[0].cpu().permute(1, 2, 0).numpy()
    rnd_np = f1_rand[0].cpu().permute(1, 2, 0).numpy()
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(f0_np)
axes[0].set_title('input t-1')
axes[0].axis('off')
axes[1].imshow(rnd_np)
axes[1].set_title('random t (simulated cut)')
axes[1].axis('off')
axes[2].imshow(bad_np)
axes[2].set_title('model output (cut input)')
axes[2].axis('off')
plt.suptitle('Scene-cut robustness probe (mismatched inputs)')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'scene_cut_probe.png'), dpi=150)
plt.show()
# Disocclusion proxy: error concentrated where |t-1 - t| is large
with torch.no_grad():
    errs, mots = [], []
    for f0, fm, f1 in tqdm(test_loader, desc='disocc-probe'):
        f0 = f0.to(device)
        fm = fm.to(device)
        f1 = f1.to(device)
        out = _fwd(f0, f1)
        pred = (out[0] if isinstance(out, (tuple, list)) else out).clamp(0, 1)
        mot = (f0 - f1).abs().mean(dim=1, keepdim=True)
        err = (pred - fm).abs().mean(dim=1, keepdim=True)
        mask = (mot > 0.15).float()
        errs.append(float((err * mask).sum() / (mask.sum() + 1e-6)))
        mots.append(float(mask.mean()))
dis_df = pd.DataFrame([{'metric': 'mean_err_in_motion_mask', 'value': round(float(np.mean(errs)), 5)}, {'metric': 'mean_motion_mask_frac', 'value': round(float(np.mean(mots)), 4)}, {'metric': 'test_psnr', 'value': round(test_psnr, 3)}])
print(dis_df.to_string(index=False))
# FlyingChairs EPE spot check (if pairs + .flo present)
epe_rows = []
try:
    shown = 0
    for (pa, pb, pf) in tqdm(chairs_pairs[:50], desc='chairs-epe'):
        cand = pa.parent / (pa.name.replace('_img1.ppm', '_flow.flo'))
        if not cand.exists():
            continue
        try:
            with Image.open(pa) as im:
                ia = im.convert('RGB')
            with Image.open(pb) as im:
                ib = im.convert('RGB')
            flow = read_flo(cand)  # [2,H,W]
            H0, W0 = flow.shape[1], flow.shape[2]
            ia_r = ia.resize((128, 128), Image.BICUBIC)
            ib_r = ib.resize((128, 128), Image.BICUBIC)
            t0 = TF.to_tensor(ia_r).unsqueeze(0).to(device)
            t1 = TF.to_tensor(ib_r).unsqueeze(0).to(device)
            with torch.no_grad():
                out = _fwd(t0, t1)
                aux = out[1] if isinstance(out, (tuple, list)) else None
            if aux is None:
                continue
            fl = aux['flow_0'][0].detach().float().cpu().numpy()
            # rescale GT flow (full-res pixels) to 128px crop space, approx by resize ratio
            sx, sy = 128.0 / W0, 128.0 / H0
            gt = np.stack([flow[0] * sx, flow[1] * sy], axis=0)
            gt_r = np.stack([np.asarray(Image.fromarray(gt[k]).resize((128, 128), Image.BILINEAR)) for k in range(2)], axis=0)
            epe = float(np.sqrt(((fl - gt_r) ** 2).sum(axis=0)).mean())
            epe_rows.append({'pair': pa.name[:24], 'epe_px': round(epe, 3)})
            shown += 1
            if shown >= 5:
                break
        except Exception as e:
            logger.warning('chairs pair skipped: %s', e)
            continue
except Exception as e:
    logger.warning('chairs EPE probe failed: %s', e)
if epe_rows:
    print(pd.DataFrame(epe_rows).to_string(index=False))
    logger.info('FlyingChairs EPE probe rows %d (untrained flow expected to be weak)', len(epe_rows))
else:
    logger.warning('no FlyingChairs .flo pairs found; EPE probe skipped (attach craljimenez/flyingchairs)')
# Vid4 held-out zero-shot (triplets from held-out files, eval only)
try:
    h_by_parent = defaultdict(list)
    for p in holdout_files:
        h_by_parent[str(p.parent)].append(p)
    h_trips = []
    for parent, files in sorted(h_by_parent.items()):
        files = sorted(files)
        for i in range(len(files) - 2):
            h_trips.append((files[i], files[i + 1], files[i + 2], parent))
    logger.info('Vid4 held-out triplets: %d', len(h_trips))
    if h_trips:
        from torch.utils.data import DataLoader as _DL
        h_ds = TripletFGDataset(h_trips[:200], crop=cfg.val_crop, is_train=False, seed=SEED)
        h_loader = _DL(h_ds, batch_size=cfg.batch_per_gpu * max(1, NGPUS), shuffle=False, num_workers=0)
        model.eval()
        tot_g, n_g = 0.0, 0
        with torch.no_grad():
            for f0, fm, f1 in tqdm(h_loader, desc='vid4-zero-shot'):
                f0, fm, f1 = f0.to(device), fm.to(device), f1.to(device)
                out = _fwd(f0, f1)
                pred = (out[0] if isinstance(out, (tuple, list)) else out).clamp(0, 1)
                tot_g += float(psnr_batch(pred, fm).mean()) * f0.shape[0]
                n_g += f0.shape[0]
        hz = tot_g / max(1, n_g)
        print(pd.DataFrame([{'metric': 'vid4_zero_shot_psnr', 'value': round(hz, 3)}, {'metric': 'test_psnr', 'value': round(test_psnr, 3)}]).to_string(index=False))
        logger.info('Vid4 zero-shot PSNR %.3f', hz)
except Exception as e:
    logger.warning('Vid4 zero-shot skipped: %s', e)


In [ ]:
logger.info('FG inference benchmark (540p and 1080p inputs)')
import torch.nn as nn
model.eval()
results = []
for (LRH, LRW) in [(540, 960), (1080, 1920)]:
    trial0 = torch.randn(1, 3, LRH, LRW, device=device)
    trial1 = torch.randn(1, 3, LRH, LRW, device=device)
    if cfg.use_channels_last and device.type == 'cuda':
        trial0 = trial0.to(memory_format=torch.channels_last)
        trial1 = trial1.to(memory_format=torch.channels_last)
    for _ in tqdm(range(10), desc='warmup-%dx%d' % (LRW, LRH)):
        with torch.no_grad():
            if USE_AMP:
                try:
                    with torch.amp.autocast('cuda', dtype=torch.float16):
                        y = model(trial0, trial1)
                except Exception:
                    y = model(trial0, trial1)
            else:
                y = model(trial0, trial1)
    if device.type == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    N = 30
    times = []
    with torch.no_grad():
        for _ in tqdm(range(N), desc='benchmark-fg-%dx%d' % (LRW, LRH)):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            if USE_AMP:
                try:
                    with torch.amp.autocast('cuda', dtype=torch.float16):
                        y = model(trial0, trial1)
                except Exception:
                    y = model(trial0, trial1)
            else:
                y = model(trial0, trial1)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append(1000.0 * (time.time() - t0))
    times = np.asarray(times)
    peak_mb = torch.cuda.max_memory_allocated() / 1e6 if device.type == 'cuda' else 0.0
    row = {'input': '%dx%d' % (LRW, LRH), 'mean_ms': round(float(times.mean()), 3), 'p50_ms': round(float(np.median(times)), 3), 'p95_ms': round(float(np.percentile(times, 95)), 3), 'fps': round(1000.0 / float(times.mean()), 1), 'peak_vram_MB': round(float(peak_mb), 1)}
    results.append(row)
    logger.info('%s: mean %.2f ms p95 %.2f ms fps %.1f vram %.1f MB', row['input'], row['mean_ms'], row['p95_ms'], row['fps'], row['peak_vram_MB'])
bench_df = pd.DataFrame(results)
bench_df['params'] = count_params(model if not isinstance(model, nn.DataParallel) else model.module)
print(bench_df.to_string(index=False))
bench_df.to_csv(str(OUT_DIR / 'benchmark_fg.csv'), index=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(bench_df['input'], bench_df['mean_ms'])
ax.set_ylabel('mean latency (ms)')
ax.set_title('NeuralFG latency by input resolution')
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'latency_hist_fg.png'), dpi=150)
plt.show()
logger.info('saved benchmark_fg.csv and latency_hist_fg.png')


In [ ]:
logger.info('NeuralFG ONNX export')
import torch.nn as nn
src = model.module if isinstance(model, nn.DataParallel) else model
try:
    base_m = getattr(getattr(src, '_orig_mod', None), '_orig_mod', None) or getattr(src, '_orig_mod', None) or (src.module if hasattr(src, 'module') else src)
    core = base_m if isinstance(base_m, NeuralFG) else src
    is_fg = isinstance(core, NeuralFG)
except Exception:
    is_fg = False
    core = src
logger.info('export source is NeuralFG: %s', is_fg)
if is_fg:
    core_cpu = core.to('cpu').eval()
    dummy0 = torch.randn(1, 3, 128, 128)
    dummy1 = torch.randn(1, 3, 128, 128)
    class FGInfer(nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m
        def forward(self, a, b):
            out = self.m(a, b)
            return out[0] if isinstance(out, (tuple, list)) else out
    wrapper = FGInfer(core_cpu).eval()
    torch.save({'model': core_cpu.state_dict()}, str(OUT_DIR / 'fg_model.pt'))
    logger.info('saved outputs_fg/fg_model.pt')
    try:
        torch.onnx.export(wrapper, (dummy0, dummy1), str(OUT_DIR / 'model_fg.onnx'), input_names=['frame_prev', 'frame_curr'], output_names=['frame_mid'], opset_version=17, dynamic_axes={'frame_prev': {2: 'H', 3: 'W'}, 'frame_curr': {2: 'H', 3: 'W'}, 'frame_mid': {2: 'H', 3: 'W'}})
        logger.info('saved outputs_fg/model_fg.onnx')
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(str(OUT_DIR / 'model_fg.onnx'), providers=['CPUExecutionProvider'])
            o = sess.run(None, {'frame_prev': dummy0.numpy(), 'frame_curr': dummy1.numpy()})[0]
            logger.info('onnxruntime check passed %s', str(o.shape))
        except Exception as e:
            logger.warning('onnx verify skipped: %s', e)
    except Exception as e:
        logger.warning('onnx export skipped: %s', e)
else:
    logger.warning('FG export skipped: compiled or wrapped model not unwrappable in this session')
print(pd.DataFrame([{'file': 'fg_model.pt', 'done': str((OUT_DIR / 'fg_model.pt').exists())}, {'file': 'model_fg.onnx', 'done': str((OUT_DIR / 'model_fg.onnx').exists())}]).to_string(index=False))


In [ ]:
logger.info('packaging all FG outputs')
files = []
for root, ds, fs in os.walk(str(OUT_DIR)):
    for f in fs:
        p = Path(root) / f
        try:
            files.append({'file': str(p.relative_to(WORKING)), 'kb': round(p.stat().st_size / 1024.0, 1)})
        except Exception:
            continue
for root, ds, fs in os.walk(str(CKPT_DIR)):
    for f in fs:
        p = Path(root) / f
        try:
            files.append({'file': str(p.relative_to(WORKING)), 'kb': round(p.stat().st_size / 1024.0, 1)})
        except Exception:
            continue
man = pd.DataFrame(sorted(files, key=lambda r: r['file']))
man.to_csv(str(OUT_DIR / 'manifest_fg.csv'), index=False)
print(man.to_string(index=False))
try:
    zp = shutil.make_archive(str(WORKING / 'neural_fg_outputs'), 'zip', str(OUT_DIR))
    logger.info('zip ready %s', zp)
except Exception as e:
    logger.warning('zip skipped: %s', e)
final_df = pd.DataFrame([{'metric': 'best_val_psnr_dB', 'value': round(float(best_row['val_psnr']), 3)}, {'metric': 'best_val_ssim', 'value': round(float(best_row['val_ssim']), 5)}, {'metric': 'test_psnr_dB', 'value': round(float(test_psnr), 3)}, {'metric': 'test_ssim', 'value': round(float(test_ssim), 5)}, {'metric': 'gain_over_avg_blend_dB', 'value': round(float(valm['gain_over_avg'].mean()), 3)}, {'metric': 'share_beats_avg_pct', 'value': round(100.0 * (valm['gain_over_avg'] > 0).mean(), 2)}, {'metric': 'mean_latency_ms_960x540', 'value': float(bench_df[bench_df['input'] == '960x540']['mean_ms'].iloc[0]) if len(bench_df) else None}, {'metric': 'p95_latency_ms_1920x1080', 'value': float(bench_df[bench_df['input'] == '1920x1080']['p95_ms'].iloc[0]) if len(bench_df) else None}, {'metric': 'params', 'value': int(count_params(model))}, {'metric': 'effective_batch', 'value': EFFECTIVE_BATCH}])
print(final_df.to_string(index=False))
final_df.to_csv(str(OUT_DIR / 'final_summary_fg.csv'), index=False)
logger.info('saved manifest_fg.csv, final_summary_fg.csv, history_fg.csv, metrics and figures')
print('key paths:')
print('- outputs: /kaggle/working/outputs_fg')
print('- checkpoints: /kaggle/working/checkpoints_fg')
print('- log: /kaggle/working/logs/train_fg.log')


## Results, limits and next steps

Read final_summary_fg.csv, val_metrics_fg.csv, training_curves_fg.png, predictions_grid_fg.png,
flow_quiver.png and latency_hist_fg.png together. A healthy run shows val PSNR above the average-blend
baseline on most triplets, falling loss curves, sharp interpolated mids and single-digit millisecond
latency at 540p.

Limits to state honestly: RGB-only training without engine motion vectors or depth, no VGG/LPIPS
perceptual term because Kaggle runs offline, and FlyingChairs EPE is a spot check only (flow heads train
without direct flow supervision). Vid4 is evaluated zero-shot and never trained on.

Next steps: attach the G-buffer pack for 10-channel conditioning, add a learned perceptual term when
online weights are allowed, and export model_fg.onnx to DirectML or ncnn for engine tests. Cite every
attached Kaggle dataset in any write-up and respect its licence (Vimeo/FlyingChairs are
research-only and must be excluded from commercial weights per RESEARCH.md section 3.7).
